<a href="https://colab.research.google.com/github/kannanrk28/Capstone_Masai_FinalProject_Kannan/blob/main/part2/Supervised_ML_Build_Evaluate_Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Task 1 : Load cleaned_data.csv from Part 1
# Feature Matrix, Regression Label, Classification label

import pandas as pd

# Load cleaned dataset
URL = "https://raw.githubusercontent.com/kannanrk28/Capstone_Masai_FinalProject_Kannan/main/dataset/cleaned_data.csv"
df_clean = pd.read_csv(URL)
# df_clean = pd.read_csv("cleaned_data.csv")

# -----------------------------
# Define Regression Target
# -----------------------------
y_reg = df_clean['revenue']

# -----------------------------
# Define Binary Classification Target
# -----------------------------
y_clf = (df_clean['revenue'] > df_clean['revenue'].median()).astype(int)

# -----------------------------
# Define Feature Matrix
# -----------------------------
# X = df_clean.drop(columns=['revenue'])
drop_cols = [
    'revenue',                 # Target variable
    'revenue_adj',             # Data leakage (derived from revenue)
    'id',                      # Noisy feature (unique identifier)
    'imdb_id',                 # Noisy feature (unique movie identifier)
    'homepage',                # Noisy feature (URL, mostly unique and many missing values)
    'overview',                # Noisy feature (free-text description)
    'tagline',                 # Noisy feature (free-text field)
    'keywords',                # Noisy feature (text with high-cardinality keywords)
    'cast',                    # Noisy feature (high-cardinality text)
    'original_title',          # Noisy feature (mostly unique movie titles)
    'popularity',              # Potential data leakage (calculated after movie release)
    'vote_average',            # Potential data leakage (available after audience ratings)
    'vote_count',              # Potential data leakage (generated after movie release)
    'budget_adj'               # Redundant feature (inflation-adjusted version of budget)
]

# Convert release_date to datetime
df_clean['release_date'] = pd.to_datetime(df_clean['release_date'])

# Extract only the month
df_clean['release_month'] = df_clean['release_date'].dt.month

# Drop release_date if it is no longer needed
df_clean.drop(columns=['release_date'], inplace=True)

X = df_clean.drop(columns=drop_cols)

print("Feature Matrix Shape :", X.shape)
print("Regression Target Shape :", y_reg.shape)
print("Classification Target Shape :", y_clf.shape)

# Display class distribution
print("\nClassification Target Distribution:")
print(y_clf.value_counts())
print(X.dtypes)

In [ ]:
# Part 2 - Task 2 : Encoding Categorical columns
import pandas as pd

# Create a copy of the feature matrix
X_encoded = X.copy()

# One-hot encode the 'genres' column
genre_dummies = X_encoded['genres'].str.get_dummies(sep='|')

# Define additional columns to drop which are categorical and not being encoded
# These columns were identified as problematic in the error trace
additional_drop_cols = ['director', 'production_companies']

# Remove the original 'genres' column and add the encoded columns
X_encoded = pd.concat(
    [X_encoded.drop(columns=['genres'] + additional_drop_cols), genre_dummies],
    axis=1
)

# Display the shape of the encoded dataset
print("Shape of Encoded Feature Matrix:")
print(X_encoded.shape)

# Display data types
print("\nData Types:")
print(X_encoded.dtypes)

# Display the first five rows
print("\nFirst Five Rows:")
print(X_encoded.head())

In [ ]:
# Part2 - Task 3
# Leak Free, Train, Test, Split and Scaling

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# -----------------------------------------
# Regression Dataset Split
# -----------------------------------------
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_encoded,
    y_reg,
    test_size=0.2,
    random_state=42
)

# -----------------------------------------
# Classification Dataset Split
# -----------------------------------------
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_encoded,
    y_clf,
    test_size=0.2,
    random_state=42
)

# -----------------------------------------
# Feature Scaling (Regression)
# -----------------------------------------
scaler_reg = StandardScaler()

# Fit only on training data
X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)

# Transform test data
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

# -----------------------------------------
# Feature Scaling (Classification)
# -----------------------------------------
scaler_clf = StandardScaler()

# Fit only on training data
X_train_clf_scaled = scaler_clf.fit_transform(X_train_clf)

# Transform test data
X_test_clf_scaled = scaler_clf.transform(X_test_clf)

# -----------------------------------------
# Verify Shapes
# -----------------------------------------
print("Regression Training Shape :", X_train_reg_scaled.shape)
print("Regression Testing Shape  :", X_test_reg_scaled.shape)

print("\nClassification Training Shape :", X_train_clf_scaled.shape)
print("Classification Testing Shape  :", X_test_clf_scaled.shape)

In [ ]:
# Part 2 - Task 4 - Regression model — Linear Regression: compare with Ridge Regression
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd

# ============================
# Linear Regression
# ============================

lr_model = LinearRegression()

# Train the model
lr_model.fit(X_train_reg_scaled, y_train_reg)

# Predictions
y_pred_reg = lr_model.predict(X_test_reg_scaled)

# Evaluation Metrics
lr_mse = mean_squared_error(y_test_reg, y_pred_reg)
lr_r2 = r2_score(y_test_reg, y_pred_reg)

print("Linear Regression Performance")
print(f"MSE : {lr_mse:.2f}")
print(f"R²  : {lr_r2:.4f}")

# ============================
# Feature Coefficients
# ============================

coefficients = pd.DataFrame({
    'Feature': X_encoded.columns,
    'Coefficient': lr_model.coef_
})

print("\nModel Coefficients")
print(coefficients)

# Top 3 features with highest absolute coefficients
top3 = coefficients.reindex(
    coefficients['Coefficient'].abs().sort_values(ascending=False).index
).head(3)

print("\nTop 3 Features (Absolute Coefficients)")
print(top3)

# ============================
# Ridge Regression
# ============================

ridge_model = Ridge(alpha=1.0)

ridge_model.fit(X_train_reg_scaled, y_train_reg)

y_pred_ridge = ridge_model.predict(X_test_reg_scaled)

ridge_mse = mean_squared_error(y_test_reg, y_pred_ridge)
ridge_r2 = r2_score(y_test_reg, y_pred_ridge)

print("\nRidge Regression Performance")
print(f"MSE : {ridge_mse:.2f}")
print(f"R²  : {ridge_r2:.4f}")

# ============================
# Comparison Table
# ============================

comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge Regression"],
    "MSE": [lr_mse, ridge_mse],
    "R²": [lr_r2, ridge_r2]
})

print("\nModel Comparison")
print(comparison)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Select numeric features only
numeric_features = X_encoded.select_dtypes(include=['int64', 'float64'])

# Correlation matrix
corr_matrix = numeric_features.corr()

# Display heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
# Part 2 - Task 5a - Classification model — Logistic Regression:
print("Training Class Distribution:")
print(y_train_clf.value_counts())

print("\nTraining Class Percentage:")
print(y_train_clf.value_counts(normalize=True) * 100)

In [ ]:

# Part 2 - Task 5a - Logistic Regression - Find best threshold for Precision and Recall and F1 scores
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# Predict probabilities
y_prob = log_model.predict_proba(X_test_clf_scaled)[:, 1]

best_threshold = 0.50
best_precision = 0
best_recall = 0
best_f1 = 0

for threshold in [i/100 for i in range(30, 71, 5)]:

    y_pred = (y_prob >= threshold).astype(int)

    precision = precision_score(y_test_clf, y_pred)
    recall = recall_score(y_test_clf, y_pred)
    f1 = f1_score(y_test_clf, y_pred)

    print(f"Threshold={threshold:.2f} "
          f"Precision={precision:.3f} "
          f"Recall={recall:.3f} "
          f"F1={f1:.3f}")

    if f1 > best_f1:
        best_threshold = threshold
        best_precision = precision
        best_recall = recall
        best_f1 = f1

print("\nBest Threshold")
print("Threshold :", best_threshold)
print("Precision :", best_precision)
print("Recall    :", best_recall)
print("F1 Score  :", best_f1)

In [ ]:

# Part 2 - Task 5a - Logistic Regression - Set customized threshold and generate AUC

from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train_clf_scaled, y_train_clf)

# Predict class labels
y_pred = log_model.predict(X_test_clf_scaled)

# Predict probabilities
y_prob = log_model.predict_proba(X_test_clf_scaled)[:, 1]

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

print("Confusion Matrix")
print(confusion_matrix(y_test_clf, y_pred))

print("\nClassification Report")
print(classification_report(y_test_clf, y_pred))

print(f"Accuracy : {accuracy_score(y_test_clf, y_pred):.4f}")

from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# ROC values
fpr, tpr, thresholds = roc_curve(y_test_clf, y_prob)

# AUC
auc = roc_auc_score(y_test_clf, y_prob)

plt.figure(figsize=(8,6))

plt.plot(
    fpr,
    tpr,
    label=f"AUC = {auc:.3f}",
    linewidth=2
)

plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")

plt.legend()

plt.grid(True)

plt.show()

print(f"AUC Score : {auc:.4f}")

In [ ]:
# Part 2 - 5b -
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# Threshold values
thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.60, 0.70]

results = []

for threshold in thresholds:

    # Convert probabilities to class labels
    y_pred_threshold = (y_prob >= threshold).astype(int)

    # Calculate evaluation metrics
    precision = precision_score(y_test_clf, y_pred_threshold)
    recall = recall_score(y_test_clf, y_pred_threshold)
    f1 = f1_score(y_test_clf, y_pred_threshold)

    results.append([
        threshold,
        precision,
        recall,
        f1
    ])

# Display results as a table
threshold_results = pd.DataFrame(
    results,
    columns=['Threshold', 'Precision', 'Recall', 'F1 Score']
)

print(threshold_results)

In [ ]:
# Part 2 - Task 6 - Regularization experiment on Logistic Regression:
LogisticRegression(C=1.0, max_iter=1000)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import pandas as pd

# -------------------------------
# Baseline Logistic Regression
# -------------------------------
baseline_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42
)

baseline_model.fit(X_train_clf_scaled, y_train_clf)

baseline_prob = baseline_model.predict_proba(X_test_clf_scaled)[:, 1]
baseline_pred = baseline_model.predict(X_test_clf_scaled)

# -------------------------------
# Strong Regularization
# -------------------------------
regularized_model = LogisticRegression(
    C=0.01,
    max_iter=1000,
    random_state=42
)

regularized_model.fit(X_train_clf_scaled, y_train_clf)

regularized_prob = regularized_model.predict_proba(X_test_clf_scaled)[:, 1]
regularized_pred = regularized_model.predict(X_test_clf_scaled)

# -------------------------------
# Comparison Table
# -------------------------------
comparison = pd.DataFrame({
    "Model": ["Baseline (C=1.0)", "Regularized (C=0.01)"],
    "Precision": [
        precision_score(y_test_clf, baseline_pred),
        precision_score(y_test_clf, regularized_pred)
    ],
    "Recall": [
        recall_score(y_test_clf, baseline_pred),
        recall_score(y_test_clf, regularized_pred)
    ],
    "F1 Score": [
        f1_score(y_test_clf, baseline_pred),
        f1_score(y_test_clf, regularized_pred)
    ],
    "AUC": [
        roc_auc_score(y_test_clf, baseline_prob),
        roc_auc_score(y_test_clf, regularized_prob)
    ]
})

print(comparison)

In [ ]:
# Bootstrap confidence interval for AUC difference
import numpy as np
from sklearn.metrics import roc_auc_score

# Set random seed for reproducibility
np.random.seed(42)

n_bootstrap = 500
auc_differences = []

# Convert y_test_clf to array if needed
y_test_array = np.array(y_test_clf)

for i in range(n_bootstrap):
    # Sample test row indices with replacement
    sample_idx = np.random.choice(
        len(y_test_array),
        size=len(y_test_array),
        replace=True
    )

    # Bootstrap sample
    y_sample = y_test_array[sample_idx]
    baseline_sample_prob = baseline_prob[sample_idx]
    regularized_sample_prob = regularized_prob[sample_idx]

    # Skip sample if it contains only one class
    if len(np.unique(y_sample)) < 2:
        continue

    # Compute AUC for both models
    auc_baseline = roc_auc_score(y_sample, baseline_sample_prob)
    auc_regularized = roc_auc_score(y_sample, regularized_sample_prob)

    # Difference: C=1.0 AUC - C=0.01 AUC
    auc_diff = auc_baseline - auc_regularized
    auc_differences.append(auc_diff)

# Convert to numpy array
auc_differences = np.array(auc_differences)

# Mean difference and 95% confidence interval
mean_diff = auc_differences.mean()
ci_lower, ci_upper = np.percentile(auc_differences, [2.5, 97.5])

print(f"Mean AUC Difference: {mean_diff:.4f}")
print(f"95% Bootstrap CI: [{ci_lower:.4f}, {ci_upper:.4f}]")